### JB Hifi

In [26]:
import time, csv, json, re, requests, os, random, shutil, glob, warnings
import pandas as pd
import numpy as np
import unicodedata
from zoneinfo import ZoneInfo
from html import unescape
from datetime import datetime, timezone
from pathlib import Path
from html import unescape
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException, NoSuchElementException, TimeoutException


REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = Path(os.environ.get("JB_HIFI_DATA_DIR", REPO_ROOT.parent / "jb_hifi_private_data")).resolve()
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
BACKUP_DIR = DATA_DIR / "backups"
DAILY_DIR = RAW_DIR / "jb_daily_data"
RESULTS_SPECS = RAW_DIR / "results_specs.csv"
RESULTS_WITH_BRAND = PROCESSED_DIR / "results_with_brand.csv"
RESULTS_TAGGED = PROCESSED_DIR / "results_tagged.csv"
RESULTS_SPECS_CLEANED = PROCESSED_DIR / "results_specs_cleaned_all.csv"
MASTER_BACKUP_DIR = BACKUP_DIR / "jb_backup_data"

LAPTOPS_URL = "https://www.jbhifi.com.au/collections/computers-tablets/laptops?hitsPerPage=100"

LISTING_CORE_COLUMNS = [
    "DateCollected", "Brand", "Title", "Price", "FullPrice", "Link", "ImageURL", "Rating", "NumRating",
]
DAILY_APPEND_KEY_COLUMNS = ["DateCollected", "Title"]
SPECS_TITLE_COLUMN = "Title"


def missing_columns(columns, required_columns):
    return [column for column in required_columns if column not in columns]


def require_columns(columns, required_columns, dataset_name):
    missing = missing_columns(columns, required_columns)
    if missing:
        raise ValueError(f"{dataset_name} is missing required columns: {missing}")


In [27]:
# FUNCTIONS
# These selectors depend on JB Hi-Fi's storefront markup and may break after a site redesign.
LOAD_MORE_BUTTON_CLASS = "load-more-button"
PRODUCT_CARD_CLASS = "ProductCard"
PRODUCT_TITLE_SELECTOR = "[data-testid='product-card-title']"
PRODUCT_LINK_SELECTOR = "a.ProductCard_imageLink"
PRICE_SYMBOL_SELECTOR = "span[class*='PriceTag_symbol']"
PRICE_AMOUNT_SELECTOR = "span[class*='PriceTag_actual']"
PRODUCT_IMAGE_SELECTOR = "img"
FULL_PRICE_SELECTOR = "span[class*='PriceTag_symbolHeader'] + span"
RATING_SELECTOR = "[data-testid='product-card-reviews'] button._6zw1gn1 ._6zw1gna"
RATING_COUNT_SELECTOR = "div[class*='_6zw1gnb']"
PROMO_TAG_SELECTOR = (
    "span[data-testid='product-card-banner-tag'], "
    "span[data-testid^='product-card-promo-tag-']"
)

REQUIRED_SCRAPED_ROW_FIELDS = ("date", "title", "price", "fullprice", "link", "image", "tags")

# Setup & Utilities
def setup_driver():
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # Run Chrome without a visible window
    return webdriver.Chrome(options=options)

# Scraping Functions
def load_all_products(driver, scroll_pause=1, click_pause=2):
    """
    Clicks the 'Load More' button repeatedly until it's gone.
    Uses JS fallback if a normal click is intercepted.
    """
    while True:
        try:
            btn = driver.find_element(By.CLASS_NAME, LOAD_MORE_BUTTON_CLASS)
            
            # Scroll into view
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", btn)
            time.sleep(scroll_pause)

            # Try a normal click, else JS click
            try:
                btn.click()
            except ElementClickInterceptedException:
                driver.execute_script("arguments[0].click();", btn)

            time.sleep(click_pause)

        except NoSuchElementException:
            print("No more 'Load More' button")
            break

# Extract Product Data
def extract_product_data(tile):
    date_collected = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()

    try:
        # Core fields
        title = tile.find_element(By.CSS_SELECTOR, PRODUCT_TITLE_SELECTOR).text.strip()
        link = tile.find_element(By.CSS_SELECTOR, PRODUCT_LINK_SELECTOR).get_attribute("href")
        symbol = tile.find_element(By.CSS_SELECTOR, PRICE_SYMBOL_SELECTOR).text.strip()
        amount = tile.find_element(By.CSS_SELECTOR, PRICE_AMOUNT_SELECTOR).text.strip()
        image = tile.find_element(By.CSS_SELECTOR, PRODUCT_IMAGE_SELECTOR).get_attribute("src")
        price = f"{symbol}{amount}".strip() if amount else "N/A"

        # Price element
        try:
            fullprice = tile.find_element(
                  By.CSS_SELECTOR,
                  FULL_PRICE_SELECTOR
               ).text.strip()
            fullprice = f"{symbol}{fullprice}"
        except NoSuchElementException:
            # if there’s no “was” price, fall back to the actual amount
            fullprice = price

        
        # Ratings
        try:
            rating_txt = tile.find_element(
                By.CSS_SELECTOR,
                RATING_SELECTOR                   
            ).text.strip()
            rating = float(rating_txt)
        except Exception:
            rating = None

        # Number of ratings
        try:
            num_txt = tile.find_element(By.CSS_SELECTOR, RATING_COUNT_SELECTOR).text.strip()
            # strip non-digits and convert
            num_ratings = int(re.sub(r'\D', '', num_txt))
        except (NoSuchElementException, ValueError):
            num_ratings = None
        
        # Find all tags
        tag_spans = tile.find_elements(
            By.CSS_SELECTOR,
            PROMO_TAG_SELECTOR
        )
        tags = [t.text.strip() for t in tag_spans if t.text.strip()]
        
        return {
            "date"     : date_collected,
            "title"    : title,
            "price"    : price,
            "fullprice": fullprice,
            "link"     : link,
            "image"    : image,
            "rating"   : rating,
            "num_ratings": num_ratings,
            "tags"     : tags
        }
    except NoSuchElementException:
        # If the core fields are missing, skip this tile
        return None

def validate_scraped_rows(rows, min_rows=1):
    issues = []
    if len(rows) < min_rows:
        issues.append(f"Expected at least {min_rows} scraped rows, found {len(rows)}.")

    for index, row in enumerate(rows, start=1):
        missing = [field for field in REQUIRED_SCRAPED_ROW_FIELDS if field not in row]
        if missing:
            issues.append(f"Row {index} is missing required fields: {missing}.")
        if "tags" in row and not isinstance(row["tags"], list):
            issues.append(f"Row {index} has non-list tags: {type(row['tags']).__name__}.")

    return issues


def validate_scraped_rows_before_write(rows):
    issues = validate_scraped_rows(rows, min_rows=1)
    unsafe = [
        issue for issue in issues
        if "missing required fields" in issue or "non-list tags" in issue
    ]
    if unsafe:
        raise ValueError(f"Scraped rows are not safe to write: {unsafe}")
    for issue in issues:
        warnings.warn(issue, RuntimeWarning, stacklevel=2)

def extract_product_rows(cards, extractor=extract_product_data):
    rows = []
    skipped_count = 0
    for card in cards:
        row = extractor(card)
        if row:
            rows.append(row)
        else:
            skipped_count += 1
    return rows, skipped_count



### FETCH SPECS INFORMATION

In [28]:
def parse_specs_from_html(html):
    # Grab every window.themeConfig call
    pattern_all = r"window\.themeConfig\(\s*['\"]([^'\"]+)['\"]\s*,\s*(\{.*?\})\s*\)\s*;"
    matches = re.findall(pattern_all, html, re.DOTALL)

    # Find product.metafields
    block = None
    for key, js in matches:
        if key.strip() == "product.metafields":
            block = js
            break
    if not block:
        return {}

    # Unescape and parse JSON
    raw = unescape(block)
    try:
        data = json.loads(raw)
        specs_list = data["online_product"]["value"]["Display"]["SpecificationDetails"]
    except Exception:
        return {}

    # Flatten
    return {
        spec["Name"]: ", ".join(map(str, spec.get("Values", [])))
        for spec in specs_list
        if spec.get("Name") and spec.get("Values")
    }


def fetch_specs(product_url):
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(product_url, headers=headers, timeout=10)
        resp.raise_for_status()
    except Exception:
        return {}

    return parse_specs_from_html(resp.text)


### RESULTS.CSV

In [ ]:
def scrape_jbhifi_laptops(output_file="results.csv"):

    # Brand extraction function
    def extract_brand(title):
        brands = [
            "HP", "ASUS", "Dell", "Apple", "Lenovo", "MSI", "Acer", "Alienware",
            "Gigabyte", "Erazer", "Microsoft", "Samsung", "Leader", "LG", 
            "Aftershock", "HyperX"
        ]

        for b in brands:
            if b.lower() in title.lower(): return b
        if "proart" in title.lower(): return "ASUS"
        if "omen" in title.lower(): return "HP"
        if "lg gram" in title.lower(): return "LG"
        if "lenvo" in title.lower(): return "Lenovo"
        if "victus" in title.lower(): return "HP"
        return "Other"

    driver = setup_driver()
    try:
        driver.get(LAPTOPS_URL)
        
        load_all_products(driver)
        cards = driver.find_elements(By.CLASS_NAME, PRODUCT_CARD_CLASS)

        print(f"Found {len(cards)} products.")

        rows, skipped_count = extract_product_rows(cards)
        if skipped_count:
            warnings.warn(
                f"Skipped {skipped_count} product cards because required fields were missing.",
                RuntimeWarning,
                stacklevel=2,
            )
    finally:
        driver.quit()

    # Determine max number of tags across all rows
    max_tags = max((len(r["tags"]) for r in rows), default=0)

    # CSV header
    core_cols = ["DateCollected", "Brand", "Title", "Price", "FullPrice", "Link", "ImageURL", "Rating", "NumRating"]
    tag_cols  = [f"Tag{i+1}" for i in range(max_tags)]
    header    = core_cols + tag_cols

    validate_scraped_rows_before_write(rows)

    # Write CSV, padding tags lists
    with open(output_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(header)

        for r in rows:
            brand     = extract_brand(r["title"])
            core_vals = [r["date"], brand, r["title"], r["price"], r["fullprice"], r["link"], r["image"], r["rating"], r["num_ratings"]]
            padded    = r["tags"] + [""]*(max_tags - len(r["tags"]))
            writer.writerow(core_vals + padded)

    print(f"Data saved to {output_file}!")

### SPECS.CSV

In [30]:
def update_specs(core_file, specs_file=RESULTS_SPECS, limit=None, delay=1, jitter=0.5):
    """
    Read results.csv to get Titles & Links.
    Read existing results_specs to skip existing Titles.
    Fetch specs for new Titles, then rewrite specs_file
    with all Titles + complete set of spec columns.
    """
    # Load core data
    with open(core_file, newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        core_rows = list(reader)
    if core_rows:
        require_columns(list(core_rows[0].keys()), [SPECS_TITLE_COLUMN, "Link"], "core listings")

    # map Title -> Link
    title_to_link = {
        unicodedata.normalize('NFKC', unescape(r["Title"].strip().lower())): r["Link"]
        for r in core_rows
    }

    # Load existing specs
    existing = {}
    if os.path.exists(specs_file):
        with open(specs_file, newline='', encoding='utf-8-sig') as f:
            rd = csv.DictReader(f)

            # Normalize headers
            rd.fieldnames = [fn.lstrip('\ufeff') for fn in rd.fieldnames]
            
            for row in rd:
                title = unicodedata.normalize('NFKC', unescape(row["Title"].strip().lower()))
                existing[title] = {
                    k: v for k, v in row.items() 
                    if k != "Title"
                }

    # Determine which Titles need specs fetched
    new_titles = [t for t in title_to_link if t not in existing]
    
    if limit:
        new_titles = new_titles[:limit]
    print(f"Found {len(new_titles)} new laptops to fetch specs for.")

    # Fetch specs for new Titles
    for i, title in enumerate(new_titles, start=1): 
        existing[title] = fetch_specs(title_to_link[title]) 
        # pause before the next request 
        wait = delay + random.random() * jitter 
        time.sleep(wait)

    # Determine full spec columns
    all_spec_keys = set()
    for specs in existing.values():
        all_spec_keys.update(specs.keys())
    all_keys = sorted(all_spec_keys)

    # Rewrite specs CSV
    with open(specs_file, 'w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(["Title"] + all_keys)

        for title, specs in existing.items():
            clean_title = unicodedata.normalize('NFKC', unescape(title))
            row_vals = []
            for key in all_keys:
                raw = specs.get(key, "")
                txt = ", ".join(map(str, raw)) if isinstance(raw, (list, tuple)) else str(raw)
                clean = unicodedata.normalize('NFKC', unescape(txt))
                row_vals.append(clean)
            
            writer.writerow([clean_title] + row_vals)

    print(f"Specs data saved to {specs_file}")

In [31]:
today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
today

'2026-07-13'

In [32]:
if __name__ == "__main__":
    today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
    os.makedirs(DAILY_DIR, exist_ok=True)
    core_file = DAILY_DIR / f"{today}_results.csv"
    scrape_jbhifi_laptops(output_file=core_file)
    update_specs(core_file=core_file, specs_file=RESULTS_SPECS)


No more 'Load More' button
Found 306 products.
Data saved to C:\Users\Admin\Downloads\Python\jb_hifi_private_data\raw\jb_daily_data\2026-07-13_results.csv!
Found 4 new laptops to fetch specs for.
Specs data saved to C:\Users\Admin\Downloads\Python\jb_hifi_private_data\raw\results_specs.csv


### COPY TO MASTER FILE results_with_brand.csv

In [33]:
def ingest_daily(daily_file, master_file, key_cols=None, backup_dir=MASTER_BACKUP_DIR):
    key_cols = key_cols or DAILY_APPEND_KEY_COLUMNS
    df_daily  = pd.read_csv(daily_file, encoding='utf-8-sig')
    assert len(df_daily) > 0, f"{daily_file} is empty!"
    require_columns(list(df_daily.columns), LISTING_CORE_COLUMNS, "daily listings")
    require_columns(list(df_daily.columns), key_cols, "daily listings")

    df_master = pd.read_csv(master_file, encoding='utf-8-sig')
    require_columns(list(df_master.columns), LISTING_CORE_COLUMNS, "master listings")
    require_columns(list(df_master.columns), key_cols, "master listings")
    assert list(df_daily.columns) == list(df_master.columns), "Column mismatch!"

    # Dup-check on key_cols
    dupes = pd.merge(
        df_daily[key_cols].drop_duplicates(),
        df_master[key_cols].drop_duplicates(),
        on=key_cols,
        how='inner'
    )
    if not dupes.empty:
        print(f"Found {len(dupes)} duplicate rows. Aborting.")
        return

    # Backup
    today = datetime.now(ZoneInfo("Australia/Melbourne")).date().isoformat()
    bak   = f"jb_bak_{today}.csv"
    os.makedirs(backup_dir, exist_ok=True)
    bak_path = Path(backup_dir) / bak
    shutil.copy(master_file, bak_path)
    print(f"Backup saved to {bak_path}")

    old_master = len(df_master)
    print(f"Daily rows : {len(df_daily)}")
    print(f"Master rows: {old_master}")
    print(f"New rows   : {len(df_daily)} (would be appended)")

    # Append daily to master
    df_new_master = pd.concat([df_master, df_daily], ignore_index=True)

    # Write updated master
    df_new_master.to_csv(master_file, index=False, encoding='utf-8-sig')
    print(f"Appended {len(df_daily)} rows. New master row-count: {len(df_new_master)}")

    # Warning
    if len(df_new_master) != len(df_daily) + old_master:
        return "Number of rows doesn't match, DOUBLE CHECK!"
    else:
        print("Matched!")


def find_latest_daily(dir_path=DAILY_DIR, pattern="*_results.csv"):
    paths = glob.glob(str(Path(dir_path) / pattern))
    if not paths:
        raise FileNotFoundError(f"No files found in {dir_path}/{pattern}")
    return max(paths, key=os.path.getmtime)


In [34]:
if __name__ == "__main__":
    master = RESULTS_WITH_BRAND
    daily  = find_latest_daily(DAILY_DIR)
    ingest_daily(daily, master, key_cols=DAILY_APPEND_KEY_COLUMNS, backup_dir=MASTER_BACKUP_DIR)


Backup saved to C:\Users\Admin\Downloads\Python\jb_hifi_private_data\backups\jb_backup_data\jb_bak_2026-07-13.csv
Daily rows : 306
Master rows: 116797
New rows   : 306 (would be appended)
Appended 306 rows. New master row-count: 117103
Matched!


### PREPROCESSING

#### 1. Results.csv

In [35]:
def validate_tagged_listings_before_write(df):
    require_columns(
        list(df.columns),
        LISTING_CORE_COLUMNS + ["Final appear day", "Discontinued"],
        "tagged listings",
    )


def parse_price_series(series):
    return series.str.replace(r"[\$,]", "", regex=True).astype("float64")


def tag_discontinued_dataframe(df):
    df = df.copy()
    require_columns(list(df.columns), LISTING_CORE_COLUMNS, "historical listings")

    # Dates
    dates = df["DateCollected"].astype(str)
    dt_iso = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    dt_eu  = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
    # Combine, preferring ISO where available
    df["DateCollected"] = dt_iso.fillna(dt_eu)

    # Flag discontinued entries
    # Get latest scrape day in dataset
    latest_day = df["DateCollected"].max()
    # Compute last known date each product appeared
    last_seen = df.groupby("Link")["DateCollected"].max().reset_index()
    last_seen.rename(columns={"DateCollected": "Final appear day"}, inplace=True)
    df = df.merge(last_seen, on="Link", how="left")
    # Flag
    df["Discontinued"] = df["Final appear day"] < latest_day
    df["Price"] = parse_price_series(df["Price"])
    return df


def tag_discontinued_products(csv_file=RESULTS_WITH_BRAND, output_file=RESULTS_TAGGED):
    """
    Loads historical listing data, adds 'Final appear day' and 'Discontinued' flags, and saves to a new file.
    """
    df = pd.read_csv(csv_file, encoding='utf-8-sig')
    df = tag_discontinued_dataframe(df)
    validate_tagged_listings_before_write(df)

    # Save
    Path(output_file).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"Tagged file saved to: {output_file}")


In [36]:
tag_discontinued_products()

Tagged file saved to: C:\Users\Admin\Downloads\Python\jb_hifi_private_data\processed\results_tagged.csv


In [37]:
# Double Check
csv_file = pd.read_csv(RESULTS_WITH_BRAND, encoding='utf-8-sig')
output_file = pd.read_csv(RESULTS_TAGGED, encoding='utf-8-sig')
print(csv_file.shape)
print(output_file.shape)


(117103, 12)
(117103, 14)


In [13]:
# summary of category features
print(output_file.dtypes)
from IPython.display import display, HTML
display(HTML('<b>Table 1: Summary of numerical features</b>'))
output_file.describe().T

DateCollected        object
Brand                object
Title                object
Price               float64
FullPrice            object
Link                 object
ImageURL             object
Rating              float64
NumRating           float64
Tag1                 object
Tag2                 object
Tag3                 object
Final appear day     object
Discontinued           bool
dtype: object


,count,mean,std,min,25%,50%,75%,max
Price,115521.0,2540.370404,1704.721647,99.0,1499.0,2159.0,2999.0,11799.0
Rating,65815.0,4.012380,1.006989,0.0,3.7,4.1,4.8,5.0
NumRating,65815.0,12.716934,20.298533,0.0,2.0,4.0,15.0,408.0


#### 2. Results_specs.csv

In [38]:
import pandas as pd
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 3000)

df = pd.read_csv(RESULTS_SPECS, encoding='utf-8-sig')
require_columns(list(df.columns), [SPECS_TITLE_COLUMN], "raw specs")

print(df.shape)

empty = df.isna().any()
empty_col = empty[empty].index.tolist()
print(empty.sum())
# empty_col


(1642, 97)
96


In [40]:
#Remove duplicates
df["Title_lower"] = df["Title"].str.lower()
df = df.drop_duplicates(subset=["Title_lower"], keep="last")
df = df.drop(columns=["Title_lower"])
print(df.shape)

(1640, 97)


In [ ]:
# Drop columns
DROP_SPEC_COLUMNS = [
    "Backlit keyboard", "Contains button battery", "PC Gaming Device", "Microsoft Surface type",
    "Tablet Type", "Network compatibility", "Wi-Fi", "Charge port", "RAM module configuration",
    "Expandable memory slot(s)", "Expandable storage type", "Battery WHr", "Battery capacity (mAh)", "Battery life",
    "TOPS (AI metric)", "Processor Memory Cache", "Device screen size (inches)", "Response Time (ms)",
    "Aspect ratio", "Graphics memory", "Flash storage", "Ethernet / LAN ports", "Intel Evo device",
    "Internal memory", "Manufacturer's Warranty", "Memory type", "Mini HDMI ports",
    "Mouse and keyboard", "Power supply type", "Processor Clock Speed (GHz)",
    "Processor Max. Clock Speed (GHz)", "RAM speeds", "RAM type", "Refresh Rate (Hz)",
    "SSD form factor", "Surface Connect ports", "eMMC storage", "Processor Model Number",
]

In [45]:
DROP_SPEC_COLUMNS = [
    "Backlit keyboard", "Contains button battery", "PC Gaming Device", "Microsoft Surface type",
    "Tablet Type", "Network compatibility", "Wi-Fi", "Charge port", "RAM module configuration",
    "Expandable memory slot(s)", "Expandable storage type", "Battery WHr", "Battery capacity (mAh)", "Battery life",
    "TOPS (AI metric)", "Processor Memory Cache", "Device screen size (inches)", "Response Time (ms)",
    "Aspect ratio", "Graphics memory", "Flash storage", "Ethernet / LAN ports", "Intel Evo device",
    "Internal memory", "Manufacturer's Warranty", "Memory type", "Mini HDMI ports",
    "Mouse and keyboard", "Power supply type", "Processor Clock Speed (GHz)",
    "Processor Max. Clock Speed (GHz)", "RAM speeds", "RAM type", "Refresh Rate (Hz)",
    "SSD form factor", "Surface Connect ports", "eMMC storage", "Processor Model Number",
]

RTX_A_TITLE_PATTERN = re.compile(r"\brtx\s+a\s*(\d{3,4})\b", re.IGNORECASE)
RTX_TITLE_PATTERN = re.compile(r"\brtx\s*-?\s*(\d{3,4})\s*(ti)?\b", re.IGNORECASE)
TITLE_DISPLAY_SIZE_PATTERN = re.compile(r'(?<![\d.])(1[0-9](?:\.\d)?|2[0-4](?:\.\d)?)\s*(?:"|inch\b|in\b)', re.IGNORECASE)
TITLE_STORAGE_BRACKET_PATTERN = re.compile(r"\[(\d+(?:\.\d+)?)\s*(tb|gb)\]", re.IGNORECASE)
TITLE_STORAGE_SSD_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?)\s*(tb|gb)\s+ssd\b|\bssd\s*(\d+(?:\.\d+)?)\s*(tb|gb)\b",
    re.IGNORECASE,
)
TITLE_STORAGE_ALLOWED_VALUES = {"64GB", "128GB", "256GB", "512GB", "1TB", "2TB", "4TB", "6TB"}
TITLE_PROCESSOR_PATTERNS = [
    (re.compile(r"\bintel\s+core\s+ultra\s+([579])\b", re.IGNORECASE), "Intel Core Ultra {group1}"),
    (re.compile(r"\bintel\s+core\s+i([3579])\b", re.IGNORECASE), "Intel Core i{group1}"),
    (re.compile(r"\bamd\s+ryzen\s+ai\s+([579])\b", re.IGNORECASE), "AMD Ryzen AI {group1}"),
    (re.compile(r"\bamd\s+ryzen\s+([3579])\b", re.IGNORECASE), "AMD Ryzen {group1}"),
]


def validate_tagged_listings_before_write(df: pd.DataFrame) -> None:
    require_columns(
        list(df.columns),
        LISTING_CORE_COLUMNS + ["Final appear day", "Discontinued"],
        "tagged listings",
    )


def validate_clean_specs_before_write(df: pd.DataFrame) -> None:
    require_columns(list(df.columns), [SPECS_TITLE_COLUMN], "cleaned specs")


def parse_price_series(series: pd.Series) -> pd.Series:
    return series.str.replace(r"[\$,]", "", regex=True).astype("float64")


def dedupe_specs_by_title(df: pd.DataFrame, keep: str = "last") -> pd.DataFrame:
    df = df.copy()
    df["Title_lower"] = df[SPECS_TITLE_COLUMN].str.lower()
    dup_count = df.duplicated(subset=["Title_lower"],keep=keep).sum()
    print(f"Found {dup_count} duplicate rows")
    df = df.drop_duplicates(subset=["Title_lower"], keep=keep)
    return df.drop(columns=["Title_lower"])


def mark_new_specs_rows(
    df: pd.DataFrame,
    old: pd.DataFrame | None = None,
    keep_existing_values: bool = True,
) -> pd.DataFrame:
    df = df.copy()
    if keep_existing_values and old is not None:
        require_columns(list(old.columns), [SPECS_TITLE_COLUMN], "existing cleaned specs")
        old_keys = set(old[SPECS_TITLE_COLUMN].str.lower())
        df["is_new"] = ~df[SPECS_TITLE_COLUMN].str.lower().isin(old_keys)
    else:
        df["is_new"] = True
    return df


def normalize_copilot_pc_column(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Copilot+ PC" in df.columns:
        copilot = df["Copilot+ PC"]
        df["Copilot+ PC"] = np.where(
            copilot.map(lambda value: isinstance(value, bool)),
            copilot,
            copilot.astype("string").str.contains(r"\byes\b", case=False, na=False),
        )
    return df


def normalize_product_flag_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["Gaming PC"] = df[SPECS_TITLE_COLUMN].str.contains("gaming laptop", case=False, na=False)
    if "AI features" in df.columns:
        df["AI features"] = df["AI features"].apply(clean_ai_feat)
    df["Product condition"] = df[SPECS_TITLE_COLUMN].apply(condition)
    if "Display type" in df.columns:
        df["Display type"] = df["Display type"].apply(bucket_display)
    return df


def normalize_port_count_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in ["USB Ports", "USB-C Ports"]:
        if col in df.columns:
            df[col] = df[col].fillna(0).astype(int).astype("Int64")
    return df


def normalize_operating_system_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Operating system" in df.columns:
        df["OS_norm"] = df["Operating system"].str.lower().str.strip()
        df["OS_norm"] = df["OS_norm"].replace({"macos sequioa": "macos sequoia"})
        df["Operating system"] = df["OS_norm"].apply(map_os)
        df["OS_family"] = df["Operating system"].apply(map_family)
    return df


def normalize_resolution_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Monitor resolution" in df.columns:
        if "Resolution (Pixels)" in df.columns:
            merged_res = df["Monitor resolution"].fillna(df["Resolution (Pixels)"])
            df = df.drop(columns=["Resolution (Pixels)"])
        else:
            merged_res = df["Monitor resolution"]
        df["Monitor resolution"] = merged_res.apply(clean_resolution)
    return df


def normalize_processor_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Processor Type" in df.columns:
        df["Processor Type"] = df["Processor Type"].apply(clean_processor_type)
        df["Processor brand"] = df["Processor Type"].apply(bucket_processor_brand)
    return df


def normalize_graphics_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "Graphics processor" in df.columns:
        df["Graphics processor"] = df.apply(get_gpu_bucket, axis=1)
        df["GPU brand"] = df["Graphics processor"].apply(bucket_gpu_brand)
    return df


def is_missing_value(value: object) -> bool:
    if pd.isna(value):
        return True
    return str(value).strip() == ""


def format_title_storage(amount: str, unit: str) -> str:
    number = float(amount)
    if number.is_integer():
        amount = str(int(number))
    else:
        amount = str(number).rstrip("0").rstrip(".")
    return f"{amount}{unit.upper()}"


def infer_display_size_from_title(title: object) -> object:
    match = TITLE_DISPLAY_SIZE_PATTERN.search(str(title))
    if not match:
        return pd.NA
    size = float(match.group(1))
    return int(size) if size.is_integer() else size


def infer_processor_from_title(title: object) -> object:
    text = str(title)
    for pattern, template in TITLE_PROCESSOR_PATTERNS:
        match = pattern.search(text)
        if match:
            return template.format(group1=match.group(1))
    return pd.NA


def infer_storage_from_title(title: object) -> object:
    text = str(title)
    match = TITLE_STORAGE_BRACKET_PATTERN.search(text)
    if match:
        storage = format_title_storage(match.group(1), match.group(2))
        return storage if storage in TITLE_STORAGE_ALLOWED_VALUES else pd.NA

    match = TITLE_STORAGE_SSD_PATTERN.search(text)
    if match:
        amount = match.group(1) or match.group(3)
        unit = match.group(2) or match.group(4)
        storage = format_title_storage(amount, unit)
        return storage if storage in TITLE_STORAGE_ALLOWED_VALUES else pd.NA

    return pd.NA


def apply_title_spec_fallbacks(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if SPECS_TITLE_COLUMN not in df.columns:
        return df

    if "Display size (inches)" in df.columns:
        inferred_display = df[SPECS_TITLE_COLUMN].apply(infer_display_size_from_title)
        mask = df["Display size (inches)"].map(is_missing_value) & inferred_display.notna()
        if mask.any():
            df["Display size (inches)"] = df["Display size (inches)"].astype("object")
            df.loc[mask, "Display size (inches)"] = inferred_display[mask]

    if "Processor Type" in df.columns:
        inferred_processor = df[SPECS_TITLE_COLUMN].apply(infer_processor_from_title)
        mask = df["Processor Type"].map(is_missing_value) & inferred_processor.notna()
        if mask.any():
            df.loc[mask, "Processor Type"] = inferred_processor[mask]

    inferred_storage = None
    for col in ["SSD storage", "Total Storage"]:
        if col in df.columns:
            if inferred_storage is None:
                inferred_storage = df[SPECS_TITLE_COLUMN].apply(infer_storage_from_title)
            mask = df[col].map(is_missing_value) & inferred_storage.notna()
            if mask.any():
                df.loc[mask, col] = inferred_storage[mask]

    return df


def preserve_existing_cleaned_values(df: pd.DataFrame, old: pd.DataFrame | None = None) -> pd.DataFrame:
    df = df.copy()
    if old is None:
        return df

    merged = df.merge(old, on=SPECS_TITLE_COLUMN, how="left", suffixes=("", "_old"))
    for col in df.columns:
        old_col = f"{col}_old"
        if old_col in merged.columns:
            merged[col] = merged[col].where(merged["is_new"], merged[old_col])
    return merged[df.columns]


def tag_discontinued_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    require_columns(list(df.columns), LISTING_CORE_COLUMNS, "historical listings")

    dates = df["DateCollected"].astype(str)
    dt_iso = pd.to_datetime(dates, format="%Y-%m-%d", errors="coerce")
    dt_eu = pd.to_datetime(dates, format="%d/%m/%Y", errors="coerce")
    df["DateCollected"] = dt_iso.fillna(dt_eu)

    latest_day = df["DateCollected"].max()
    last_seen = df.groupby("Link")["DateCollected"].max().reset_index()
    last_seen.rename(columns={"DateCollected": "Final appear day"}, inplace=True)

    df = df.merge(last_seen, on="Link", how="left")
    df["Discontinued"] = df["Final appear day"] < latest_day
    df["Price"] = parse_price_series(df["Price"])
    return df


def tag_discontinued_products(csv_file: str | Path, output_file: str | Path) -> Path:
    input_path = Path(csv_file)
    output_path = Path(output_file)
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    df = tag_discontinued_dataframe(df)

    validate_tagged_listings_before_write(df)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Tagged file saved to: {output_path}")
    return output_path


def clean_ai_feat(value: object) -> object:
    text = str(value).lower()
    if "copilot" in text and "keyboard" in text:
        return "Copilot keyboard key"
    return np.nan


def condition(title: object) -> str:
    text = str(title).lower() if pd.notna(title) else ""
    if "renewed" in text:
        return "Renewed"
    if "refurbished" in text:
        return "Refurbished"
    return "Other"


def bucket_display(value: object) -> object:
    if pd.isna(value):
        return np.nan
    text = str(value).lower()
    if "liquid retina xdr" in text:
        return "Liquid Retina XDR"
    if text.strip() == "liquid retina":
        return "Liquid Retina"
    if "oled" in text:
        return "OLED"
    if "mini led" in text or "miniled" in text:
        return "Mini LED"
    if "ips" in text:
        return "IPS"
    if "tn" in text:
        return "TN"
    if "pixelsense" in text:
        return "PixelSense"
    if "led" in text or "lcd" in text:
        return "LCD/LED"
    if any(item in text for item in ["anti-glare", "sva", "uwva"]):
        return "Anti-glare/SVA"
    if "wva" in text:
        return "WVA"
    return "Other"


def map_os(os_str: object) -> str:
    if not isinstance(os_str, str):
        return "Other"
    if "chrome" in os_str:
        return "Chrome OS"
    if "windows 10 pro" in os_str:
        return "Windows 10 Pro"
    if "windows 10 home" in os_str:
        return "Windows 10 Home"
    if "windows 11 pro national academic" in os_str:
        return "Windows 11 Pro National Academic"
    if "windows 11 pro" in os_str:
        return "Windows 11 Pro"
    if "windows 11 home plus" in os_str:
        return "Windows 11 Home Plus"
    if "windows 11 home s" in os_str:
        return "Windows 11 Home S"
    if "windows 11 home" in os_str:
        return "Windows 11 Home"
    if re.match(r"windows 11(\s|$)", os_str):
        return "Windows 11"
    if "macos sequoia" in os_str:
        return "macOS Sequoia"
    if "macos ventura" in os_str:
        return "macOS Ventura"
    if "macos monterey" in os_str:
        return "macOS Monterey"
    if "macos sonoma" in os_str:
        return "macOS Sonoma"
    if "macos mojave" in os_str:
        return "macOS Mojave"
    if "macos catalina" in os_str:
        return "macOS Catalina"
    if "macos sierra" in os_str:
        return "macOS Sierra"
    if "macos big sur" in os_str:
        return "macOS Big Sur"
    if os_str == "macos":
        return "macOS"
    return "Other"


def map_family(os_clean: str) -> str:
    if os_clean.startswith("Windows"):
        return "Windows"
    if os_clean.startswith("macOS"):
        return "macOS"
    if os_clean == "Chrome OS":
        return "Chrome OS"
    return "Other"


def clean_resolution(raw: object) -> object:
    if pd.isna(raw):
        return pd.NA
    text = str(raw).strip().lower()
    text = re.sub(r"(\s*\*\s*|\s*-?by-?\s*|\s*\u00d7\s*)", "x", text)
    match = re.search(r"(\d{3,4})\s*x\s*(\d{3,4})", text)
    if not match:
        match = re.fullmatch(r"(\d{3,4})\s+(\d{3,4})", text)
    if not match:
        return pd.NA
    return f"{match.group(1)} x {match.group(2)}"


def clean_processor_type(raw: object) -> object:
    if pd.isna(raw):
        return pd.NA
    text = str(raw)
    text = re.sub(r"\u00ae|\u2122|\u00c2\u00ae|\u00e2\u201e\u00a2|\(.*?\)|\[.*?\]", "", text)
    text = re.sub(r"coretm", "Core", text, flags=re.IGNORECASE)
    return re.sub(r"\s+", " ", text).strip()


def bucket_processor_brand(ptype: object) -> str:
    if isinstance(ptype, str):
        text = re.sub(r"\u00ae|\u2122|\u00c2\u00ae|\u00e2\u201e\u00a2|\(.*?\)|\[.*?\]", "", ptype.lower())
        if "intel" in text:
            return "Intel"
        if "qualcomm" in text:
            return "Qualcomm"
        if text.startswith("apple m"):
            return "Apple M"
        if text.startswith("apple a"):
            return "Apple A"
        if "ryzen" in text or "athlon" in text:
            return "AMD"
        if text.startswith("mtk"):
            return "MTK"
    return "Other"


def normalize_gpu_text(raw: object) -> str:
    text = str(raw).lower()
    text = re.sub(r"\u00ae|\u2122|(?<=[a-z])tm\b|\btm\b", "", text)
    return re.sub(r"\s+", " ", text).strip()


def infer_gpu_from_title(title: object) -> str | None:
    text = str(title)
    match = RTX_A_TITLE_PATTERN.search(text)
    if match:
        return f"NVIDIA RTX A{match.group(1)}"

    match = RTX_TITLE_PATTERN.search(text)
    if match:
        suffix = " Ti" if match.group(2) else ""
        return f"NVIDIA RTX {match.group(1)}{suffix}"

    return None


def get_gpu_bucket(row: pd.Series) -> str:
    if pd.isna(row.get("Graphics processor")) and not pd.isna(row.get("Graphics card series")):
        return row["Graphics card series"]

    title = str(row.get("Title", "")).lower()
    text = normalize_gpu_text(row.get("Graphics processor", ""))

    match = re.search(r"rtx[\s\-]?(\d{3,4}[ti]?)", text)
    if match:
        return f"NVIDIA RTX {match.group(1)}"
    match = re.search(r"geforce\s+(\d{3,4}[ti]?)", text)
    if match:
        return f"NVIDIA GeForce {match.group(1)}"
    if "iris" in text and "xe" in text:
        return "Intel Iris Xe"
    if "uhd" in text and "intel" in text:
        return "Intel UHD"
    if "hd graphics" in text or "intel hd" in text:
        return "Intel HD"
    if "pentium gold" in text:
        return "Intel Pentium Gold"
    if "arc" in text:
        return "Intel Arc"
    if "radeon" in text:
        return "AMD Radeon"
    if "adreno" in text:
        return "Qualcomm Adreno"
    if "mali" in text:
        return "ARM Mali"
    if "halo" in text:
        return "AMD Strix Halo"
    if "intel" in text and "graphic" in text:
        return "Intel Graphics"
    if "integrated" in text:
        return "Integrated"
    if re.search(r"\d+\s*core gpu", text) or "apple" in title:
        return "Apple GPU"
    inferred_gpu = infer_gpu_from_title(title)
    if inferred_gpu:
        return inferred_gpu
    return "Other"


def bucket_gpu_brand(raw: object) -> object:
    if pd.isna(raw):
        return pd.NA
    text = re.sub(r"®|™|\(.*?\)|\[.*?\]", "", str(raw).lower())
    if "intel" in text:
        return "Intel"
    if any(item in text for item in ("nvidia", "rtx", "gtx", "geforce")):
        return "NVIDIA"
    if any(item in text for item in ("amd", "radeon")):
        return "AMD"
    if "qualcomm" in text or "adreno" in text:
        return "Qualcomm"
    if "apple" in text:
        return "Apple"
    if "arm" in text or "mali" in text:
        return "ARM"
    if "integrated" in text:
        return "Integrated"
    return "Other"


def clean_specs_dataframe(
    df: pd.DataFrame,
    old: pd.DataFrame | None = None,
    keep_existing_values: bool = True,
) -> pd.DataFrame:
    df = df.copy()
    require_columns(list(df.columns), [SPECS_TITLE_COLUMN], "raw specs")
    df = dedupe_specs_by_title(df, keep="last")

    df = df.drop(columns=[col for col in DROP_SPEC_COLUMNS if col in df.columns])

    if keep_existing_values and old is not None:
        old = old.copy()

    df = mark_new_specs_rows(df, old=old, keep_existing_values=keep_existing_values)
    df = normalize_copilot_pc_column(df)
    df = normalize_product_flag_columns(df)
    df = normalize_port_count_columns(df)
    df = normalize_operating_system_columns(df)
    df = normalize_resolution_columns(df)
    df = apply_title_spec_fallbacks(df)
    df = normalize_processor_columns(df)
    df = normalize_graphics_columns(df)
    df = preserve_existing_cleaned_values(df, old=old)
    return df

def clean_specs(specs_file: str | Path, output_file: str | Path, keep_existing_values: bool = True) -> Path:
    """Clean raw specs.

    By default, existing cleaned values are preserved for previously seen titles.
    Set `keep_existing_values=False` to re-apply cleaning rules to every row.
    """
    specs_path = Path(specs_file)
    output_path = Path(output_file)

    df = pd.read_csv(specs_path, encoding="utf-8-sig")
    old = None
    if keep_existing_values and output_path.exists():
        old = pd.read_csv(output_path, encoding="utf-8-sig")

    df = clean_specs_dataframe(df, old=old, keep_existing_values=keep_existing_values)
    validate_clean_specs_before_write(df)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"Cleaned specs saved to {output_path}")
    return output_path

In [ ]:
clean_specs(RESULTS_SPECS,RESULTS_SPECS_CLEANED,keep_existing_values=True)

Found 2 duplicate rows
Cleaned specs saved to C:\Users\Admin\Downloads\Python\jb_hifi_private_data\processed\results_specs_cleaned_all.csv


WindowsPath('C:/Users/Admin/Downloads/Python/jb_hifi_private_data/processed/results_specs_cleaned_all.csv')

In [44]:
# Double check
results_specs = pd.read_csv(RESULTS_SPECS, encoding='utf-8-sig')
results_specs_cleaned_all = pd.read_csv(RESULTS_SPECS_CLEANED, encoding='utf-8-sig')
print(results_specs.shape)
print(results_specs_cleaned_all.shape)

(1642, 97)
(1640, 64)


In [ ]:
# Check for constant features
df = results_specs_cleaned_all
constant = df.nunique() == 1
print(f'The number of constant features within the dataset is {constant.sum()}.')
constant_col = constant[constant].index.tolist()
for c in constant_col:
    print(f"Constant col: {c}")
    print(df[c].unique())
    print('')

In [ ]:
print(df.dtypes.to_string())

In [ ]:
obj_cols = [
    col for col in df.select_dtypes(include="object").columns.tolist()
    if col != "Title"
]
for c in obj_cols:
    print(f"Unique value for '{c}'")
    print(df[c].unique())
    print('')